# 🩺 GluCare AI - Data Cleaning & Ground Truth Labeling Pipeline
### NHANES 2021–2023 Raw Data to Clean Clinical Model Dataset (Step 1 s/d Step 6)

Notebook ini mendokumentasikan seluruh tahapan kerja data engineering GluCare AI:
- **Step 1**: Inisialisasi arsitektur direktori repositori `glucare_ai/`.
- **Step 2**: Akuisisi 8 domain file mentah NHANES (`DEMO`, `BMX`, `BPXO`, `GLU`, `HDL`, `TRIGLY`, `GHB`, `FASTQX`).
- **Step 3**: Penggabungan tabel multi-domain berbasis `SEQN`.
- **Step 4**: Clinical feature engineering (MAP, rasio TG/HDL, rata-rata tekanan darah).
- **Step 5**: Pembersihan data (validasi puasa 8–24 jam, eksklusi kehamilan, segmentasi usia 20–40 tahun, 0 nulls).
- **Step 6**: **Penentuan Ground Truth Klinis (Opsi A)**: Menetapkan `label_prediabetes = 1` (Prediabetes), `label_prediabetes = 0` (Bukan Prediabetes), dan mengeksklusi kasus diabetes murni agar AI fokus pada skrining dini prediabetes.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Tambahkan root glucare_ai ke sys.path
BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

RAW_DIR = BASE_DIR / 'data' / 'raw' / 'nhanes_2021_2023'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

print(f'Raw data directory: {RAW_DIR.resolve()}')
print(f'Processed data directory: {PROCESSED_DIR.resolve()}')

## Step 1 & 2: Membaca File Mentah SAS XPT (8 Modul NHANES)

In [ ]:
from src.data_preprocessing import load_raw_nhanes_data

raw_dfs = load_raw_nhanes_data(RAW_DIR)

## Step 3: Penggabungan Tabel Berdasarkan `SEQN` (Inner Join)

In [ ]:
from src.data_preprocessing import merge_nhanes_tables

merged_df = merge_nhanes_tables(raw_dfs)
print(f'Total baris gabungan: {len(merged_df):,}')
merged_df.head()

## Step 4: Clinical Feature Engineering
- Tekanan Darah Rata-rata Sistolik & Diastolik
- Mean Arterial Pressure (MAP)
- Rasio Trigliserida / HDL (TG/HDL Ratio)
- Fallback Perhitungan BMI

In [ ]:
from src.data_preprocessing import engineer_clinical_features

engineered_df = engineer_clinical_features(merged_df)
cols = ['seqn', 'age', 'gender', 'bmi', 'bp_systolic', 'bp_diastolic', 'map_pressure', 'glucose_fasting', 'hdl', 'triglycerides', 'tg_hdl_ratio', 'hba1c', 'ground_truth_status', 'label_prediabetes']
engineered_df[cols].head()

## Step 5 & Step 6: Data Cleaning, Segmentasi Usia 20–40 Tahun, dan Ground Truth (Opsi A)
- Validasi puasa (8–24 jam)
- Eksklusi kehamilan (`RIDEXPRG == 1`)
- Filter usia target GluCare: 20–40 tahun
- **Opsi A**: Eksklusi peserta diabetes murni agar AI berfokus murni pada pemisahan individu sehat (`label = 0`) dan prediabetes (`label = 1`).

In [ ]:
from src.data_preprocessing import clean_dataset

# Dataset Target GluCare Usia 20-40 Tahun (Opsi A)
target_20_40 = clean_dataset(
    engineered_df,
    min_age=20,
    max_age=40,
    exclude_pregnant=True,
    exclude_diabetes=True, # Opsi A
    min_fasting_hours=8.0
)
target_20_40.to_csv(PROCESSED_DIR / 'nhanes_glucare_target_20_40.csv', index=False)

# Dataset Dewasa Lengkap (Usia >= 20 Tahun, Komparatif)
cleaned_adult = clean_dataset(
    engineered_df,
    min_age=20,
    max_age=None,
    exclude_pregnant=True,
    exclude_diabetes=False,
    min_fasting_hours=8.0
)
cleaned_adult.to_csv(PROCESSED_DIR / 'nhanes_glucare_cleaned_adult.csv', index=False)

## Visualisasi Distribusi Ground Truth Step 6

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=target_20_40, x='ground_truth_status', palette=['#3182CE', '#E53E3E'])
plt.title(f'Distribusi Ground Truth Step 6 (Target Usia 20-40 Tahun)\nTotal: {len(target_20_40):,} Responden (Opsi A)')
plt.xlabel('Ground Truth Status')
plt.ylabel('Jumlah Responden')
for p in ax.patches:
    h = p.get_height()
    ax.annotate(f'{int(h)} ({(h/len(target_20_40))*100:.1f}%)',
                (p.get_x() + p.get_width() / 2., h),
                ha='center', va='bottom', xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.show()

## Ringkasan Akhir Dataset Step 6

In [ ]:
print('=== Info Dataset Step 6 (Target Usia 20-40 Tahun) ===')
print(target_20_40.info())
print('\n=== Frekuensi Label Prediabetes ===')
print(target_20_40['label_prediabetes'].value_counts())